# Análise de qualidade — Bronze `novo_caged_movimentacoes`

Perfil de dados da camada Bronze do Novo Caged para identificar tratamentos necessários na construção da camada Silver. As células abaixo fazem varreduras completas na tabela (259,5 mi de linhas) — execute sob demanda; o pipeline Bronze → Silver fica no notebook `020_bronze_to_silver`.

## Análise de qualidade — Bronze `novo_caged_movimentacoes`

Levantamento dos tratamentos necessários antes de criar a tabela Delta da camada **Silver**:

1. **Perfil geral** — nulos, vazios, espaços excedentes e coerência das competências
2. **Domínios categóricos** — valores distintos das colunas de baixa cardinalidade
3. **Colunas numéricas** — formatos (vírgula decimal), faixas e outliers
4. **De-para** — códigos sem correspondência nas tabelas `ref_*` da bronze
5. **Duplicatas e cobertura** — linhas por competência e duplicidades de negócio

In [0]:
# 1) Perfil geral: nulos, vazios, espaços excedentes e coerência de competências
import pandas as pd

TBL = "workspace.bronze.novo_caged_movimentacoes"
campos = [f.name for f in spark.table(TBL).schema.fields]
cols_negocio = [c for c in campos if not c.startswith('_')]

exprs = ["count(*) AS total_linhas"]
registros = []  # (coluna, idx_nulos, idx_vazios, idx_espacos)
for c in cols_negocio:
    b = len(exprs)
    exprs.append(f"sum(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS m{b}")
    exprs.append(f"sum(CASE WHEN `{c}` IS NULL OR trim(`{c}`) = '' THEN 1 ELSE 0 END) AS m{b+1}")
    exprs.append(f"sum(CASE WHEN `{c}` IS NOT NULL AND `{c}` != trim(`{c}`) THEN 1 ELSE 0 END) AS m{b+2}")
    registros.append((c, b, b + 1, b + 2))

extras = {
    'competênciamov != _competencia_referencia': "sum(CASE WHEN `competênciamov` != `_competencia_referencia` THEN 1 ELSE 0 END)",
    'competênciadec != competênciamov': "sum(CASE WHEN `competênciadec` != `competênciamov` THEN 1 ELSE 0 END)",
    'competênciamov fora do formato AAAAMM': "sum(CASE WHEN `competênciamov` NOT RLIKE '^[0-9]{6}$' THEN 1 ELSE 0 END)",
    'competênciadec fora do formato AAAAMM': "sum(CASE WHEN `competênciadec` NOT RLIKE '^[0-9]{6}$' THEN 1 ELSE 0 END)",
}
for i, e in enumerate(extras.values()):
    exprs.append(f"{e} AS e{i}")

linha = spark.sql(f"SELECT {', '.join(exprs)} FROM {TBL}").collect()[0]
total = linha['total_linhas']

dados = [{
    'coluna': c,
    'nulos': linha[f'm{i0}'],
    'vazios': linha[f'm{i1}'],
    'espacos_excedentes': linha[f'm{i2}'],
    'pct_vazios_ou_nulos': round(100 * linha[f'm{i1}'] / total, 4),
} for c, i0, i1, i2 in registros]
display(pd.DataFrame(dados))

print(f"\nTotal de linhas: {total:,}")
for i, nome in enumerate(extras):
    v = linha[f'e{i}']
    print(f"{nome}: {v:,} ({100 * v / total:.4f}%)")

In [0]:
# 2) Domínios das colunas categóricas — varredura única com GROUPING SETS
TBL = "workspace.bronze.novo_caged_movimentacoes"

cats = ['região', 'uf', 'sexo', 'raçacor', 'categoria', 'graudeinstrução', 'tipoempregador',
        'tipodedeficiência', 'tipomovimentação', 'tipoestabelecimento', 'indtrabintermitente',
        'indtrabparcial', 'indicadoraprendiz', 'indicadordeforadoprazo', 'unidadesaláriocódigo',
        'origemdainformação', 'saldomovimentação']

rotulo = "CASE " + " ".join(f"WHEN grouping(`{c}`) = 0 THEN '{c}'" for c in cats) + " END"
valor = "CASE " + " ".join(f"WHEN grouping(`{c}`) = 0 THEN `{c}`" for c in cats) + " END"
conjuntos = ", ".join(f"(`{c}`)" for c in cats)

df_cats = spark.sql(f"""
  SELECT {rotulo} AS coluna, {valor} AS valor, count(*) AS qtd
  FROM {TBL}
  GROUP BY GROUPING SETS ({conjuntos})
""")
display(df_cats.orderBy("coluna", "qtd", "valor"))

In [0]:
# 3) Colunas numéricas: formato (vírgula decimal), faixas e outliers
import pandas as pd

TBL = "workspace.bronze.novo_caged_movimentacoes"
nums = ['idade', 'horascontratuais', 'salário', 'valorsaláriofixo', 'saldomovimentação',
        'tamestabjan', 'unidadesaláriocódigo']

exprs, registros = [], []
for c in nums:
    d = f"cast(replace(`{c}`, ',', '.') as double)"
    b = len(exprs)
    exprs += [
        f"sum(CASE WHEN `{c}` RLIKE '^-?[0-9]+$' THEN 1 ELSE 0 END) AS m{b}",
        f"sum(CASE WHEN `{c}` RLIKE '^-?[0-9]+,[0-9]{{2}}$' THEN 1 ELSE 0 END) AS m{b+1}",
        f"sum(CASE WHEN `{c}` IS NOT NULL AND NOT (`{c}` RLIKE '^-?[0-9]+$' OR `{c}` RLIKE '^-?[0-9]+,[0-9]{{2}}$') THEN 1 ELSE 0 END) AS m{b+2}",
        f"sum(CASE WHEN `{c}` IS NULL OR trim(`{c}`) = '' THEN 1 ELSE 0 END) AS m{b+3}",
        f"min({d}) AS m{b+4}",
        f"approx_percentile({d}, 0.5) AS m{b+5}",
        f"approx_percentile({d}, 0.99) AS m{b+6}",
        f"max({d}) AS m{b+7}",
    ]
    registros.append((c, b))

linha = spark.sql(f"SELECT {', '.join(exprs)} FROM {TBL}").collect()[0]

dados = []
for c, b in registros:
    dados.append({
        'coluna': c,
        'inteiros': linha[f'm{b}'],
        'virgula_2dec': linha[f'm{b+1}'],
        'outro_formato': linha[f'm{b+2}'],
        'nulos_ou_vazios': linha[f'm{b+3}'],
        'minimo': linha[f'm{b+4}'],
        'mediana': linha[f'm{b+5}'],
        'pct99': linha[f'm{b+6}'],
        'maximo': linha[f'm{b+7}'],
    })
display(pd.DataFrame(dados))

In [0]:
# 4) De-para: códigos da bronze sem correspondência nas tabelas de referência (ref_*)
TBL = "workspace.bronze.novo_caged_movimentacoes"

depara = {
    'região': 'ref_regiao',
    'uf': 'ref_uf',
    'município': 'ref_municipio',
    'seção': 'ref_secao',
    'subclasse': 'ref_subclasse',
    'cbo2002ocupação': 'ref_cbo_ocupacao',
    'categoria': 'ref_categoria',
    'graudeinstrução': 'ref_grau_instrucao',
    'tipomovimentação': 'ref_tipo_movimentacao',
    'tipoestabelecimento': 'ref_tipo_estabelecimento',
}

subconsultas = []
for c, r in depara.items():
    subconsultas.append(f"""
      SELECT '{c}' AS coluna,
             (SELECT count(*) FROM workspace.bronze.{r}) AS codigos_na_ref,
             count(DISTINCT b.`{c}`) AS codigos_na_bronze,
             count(DISTINCT CASE WHEN r.codigo IS NULL THEN b.`{c}` END) AS codigos_sem_match,
             count(CASE WHEN r.codigo IS NULL THEN 1 END) AS linhas_afetadas
      FROM {TBL} b
      LEFT JOIN workspace.bronze.{r} r ON b.`{c}` = r.codigo""")

df_depara = spark.sql(" UNION ALL ".join(subconsultas))
display(df_depara)

In [0]:
# 5) Cobertura por competência e duplicatas (escopo: última competência carregada)
TBL = "workspace.bronze.novo_caged_movimentacoes"

ultima = spark.sql(f"SELECT max(`_competencia_referencia`) AS c FROM {TBL}").collect()[0]['c']
print(f"Última competência carregada: {ultima}\n")

# Cobertura: linhas e janela de ingestão por competência (deve fechar 68 competências)
display(spark.sql(f"""
  SELECT `_competencia_referencia` AS competencia, count(*) AS linhas,
         min(`_data_ingestao`) AS inicio_ingestao, max(`_data_ingestao`) AS fim_ingestao
  FROM {TBL}
  GROUP BY `_competencia_referencia`
  ORDER BY competencia"""))

# Duplicatas de negócio na última competência (comparando todas as colunas de negócio)
cols_negocio = [f.name for f in spark.table(TBL).schema.fields if not f.name.startswith('_')]
cols_lista = ", ".join(f"`{c}`" for c in cols_negocio)

display(spark.sql(f"""
  SELECT count(*) AS total_linhas, count(DISTINCT {cols_lista}) AS linhas_distintas
  FROM {TBL} WHERE `_competencia_referencia` = '{ultima}'"""))

display(spark.sql(f"""
  SELECT {cols_lista}, count(*) AS qtd_repeticoes
  FROM {TBL} WHERE `_competencia_referencia` = '{ultima}'
  GROUP BY {cols_lista} HAVING count(*) > 1
  ORDER BY qtd_repeticoes DESC"""))